In [1]:
import numpy as np
import pandas as pd

from Mixers import *

import matplotlib.pyplot as plt
import plotly.express as px

---

In [2]:
np.set_printoptions(precision=2)
pd.options.display.precision = 2

plt.style.use('default')  # dark_background
plt.rcParams['figure.dpi'] = 250

---
#### Comparing mixing times

In [3]:
# === Parameters === #
path = 'mt_data/'
cols = ['n', 'i', 'j'] + [round(i, 2) for i in np.arange(.25, .96, .05)[::-1].tolist()]


# === Loading every dataset === #
files = ! ls $path
datasets = [pd.read_csv(f'{path}{file}', delimiter=',', names=cols) for file in files]


# === Selecting the corect mixing times === #
for idx in range(len(files)):
    dataset = datasets[idx]
    
    temp = pd.DataFrame()
    for n in dataset.n.unique():
        supremum = dataset[dataset.n == n][.25].max()
        temp = temp.append(dataset[(dataset.n == n) & (dataset[.25] == supremum)])
        temp['mc'] = files[idx].split('.')[0]
    
    datasets[idx] = temp.drop(['i', 'j'], axis=1)


# === Creating the dataset for animation === #
data = pd.concat(datasets)

data_animation = pd.DataFrame(columns=['n', 'tvd', 'mc', 't'])
for col in cols[3:]:
    temp = data[['n', 'mc', col]]
    temp.loc[:,('tvd')] = col
    
    temp.columns = ['n', 'mc', 't', 'tvd']
    
    data_animation = data_animation.append(temp, ignore_index=True)


# === Cleaning up === #
del datasets, data, temp

/opt/miniconda3/envs/mixing_times/lib/python3.9/site-packages/pandas/core/indexing.py:1667: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = value


In [11]:
fig = px.line(data_animation, x='n', y='t', 
              animation_frame='tvd', color='mc', line_shape='linear', 
              log_x=True, log_y=True, 
              range_x=[1,1000], range_y=[1,100000], 
              hover_data={'n':False, 'tvd':False, 'mc':False}, 
              labels={'n': 'Side length of the torus [log n]', 
                      't': 'Mixing time [log τ(ε)]', 
                      'tvd': 'Total variation<br>distance [ε]', 
                      'mc':'Markov strategy'
                     }
             )


fig.update_layout(hovermode='x unified')

fig['layout'].pop('updatemenus')
fig.show()

---
---
---